# Pair-K comparison

This notebook prepares a simpler `k=1`, `k=2`, `k=3` pair-k comparison for `ao-soul` only.

Steps:
1. filter the existing `pairwise_labels.jsonl` to `ao-soul`
2. create one fixed `ao-soul` manifest split from those existing ranks
3. treat that full split as `k=3`
4. simulate `k=2` and `k=1` by reducing connectivity on the observed `train` and `val` graphs
5. keep `test` fixed across all three settings

So this is not replaying the original pair generator. It is simulating lower connectivity on the ranked graph we already have.


## Step 1. Load the existing pairwise labels and keep only `ao-soul`


In [1]:
from __future__ import annotations

import hashlib
import shutil
from pathlib import Path

import pandas as pd
from IPython.display import display

from aitraf.data_ops.create_pairwise_manifests import (
    PairwiseManifestBuildConfig,
    PairwiseTaskConfig,
    create_pairwise_manifests,
)
from aitraf.datasets.score_prediction_rank import ScorePredictionRankDataset

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
EXPERIMENT_DIR = DATA_DIR / "experiments" / "pairs_k_comparison"
PAIRWISE_LABELS_DIR = EXPERIMENT_DIR / "pairwise_labels"
MANIFESTS_DIR = EXPERIMENT_DIR / "manifests"
PAIRWISE_LABELS_PATH = DATA_DIR / "pairwise_labels.jsonl"
LABELS_PATH = DATA_DIR / "labels.jsonl"
TRICK = "ao-soul"
PAIRWISE_LABEL_COLS = ("left", "right", "pref")
LABEL_COLS = ("video", "trick")
SPLIT_SEED = 42
EDGE_ORDER_SEED = 42
VAL_RATIO = 0.1
TEST_RATIO = 0.1
K_VALUES = (1, 2, 3)

PAIRWISE_LABELS_DIR.mkdir(parents=True, exist_ok=True)
MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)

pairwise_labels_df = pd.read_json(PAIRWISE_LABELS_PATH, lines=True)
ao_soul_pairwise_labels_df = pairwise_labels_df.loc[
    pairwise_labels_df["trick"] == TRICK
].reset_index(drop=True)

ao_soul_pairwise_labels_path = PAIRWISE_LABELS_DIR / "ao_soul_full.jsonl"
ao_soul_pairwise_labels_df.to_json(
    ao_soul_pairwise_labels_path,
    orient="records",
    lines=True,
    force_ascii=False,
)

display(
    pd.DataFrame(
        [
            {
                "trick": TRICK,
                "pairwise_labels": len(ao_soul_pairwise_labels_df),
                "output_path": str(ao_soul_pairwise_labels_path),
            }
        ]
    )
)

,trick,pairwise_labels,output_path
0,ao-soul,313,/workspace/data/experiments/pairs_k_comparison...


## Step 2. Create one fixed `ao-soul` manifest split

This gives the full `k=3` world. We will only simulate smaller `k` values on the observed `train` and `val` splits.


In [2]:
create_pairwise_manifests(
    PairwiseManifestBuildConfig(
        output_dir=MANIFESTS_DIR,
        val_ratio=VAL_RATIO,
        test_ratio=TEST_RATIO,
        split_seed=SPLIT_SEED,
        force=True,
        tasks=(
            PairwiseTaskConfig(
                name="k3",
                pairwise_labels_path=ao_soul_pairwise_labels_path,
                labels_path=LABELS_PATH,
                pairwise_labels_required_cols=PAIRWISE_LABEL_COLS,
                labels_required_cols=LABEL_COLS,
                manifests_dir=MANIFESTS_DIR / "k3",
                vocab_path=MANIFESTS_DIR / "k3" / "vocab.json",
                stratify_col=None,
            ),
        ),
    )
)

k3_splits = {
    split: pd.DataFrame(
        ScorePredictionRankDataset(
            manifests_dir=MANIFESTS_DIR / "k3", split=split
        ).manifest_rows()
    )
    for split in ["train", "val", "test"]
}

display(
    pd.DataFrame([{"split": split, "rows": len(df)} for split, df in k3_splits.items()])
)

2026-04-01 09:19:51.611 | WARNING  | aitraf.data_ops.create_pairwise_manifests:_build_pairwise_manifest_df:203 - Dropping 3 pairwise label rows without a selected preference
2026-04-01 09:19:51.621 | INFO     | aitraf.data_ops.create_pairwise_manifests:_write_vocab:334 - Wrote categorical vocab to /workspace/data/experiments/pairs_k_comparison/manifests/k3/vocab.json
2026-04-01 09:19:51.625 | INFO     | aitraf.data_ops.create_pairwise_manifests:_build_task_manifests:161 - Task 'k3' wrote /workspace/data/experiments/pairs_k_comparison/manifests/k3/train.jsonl (247 rows)
2026-04-01 09:19:51.626 | INFO     | aitraf.data_ops.create_pairwise_manifests:_build_task_manifests:161 - Task 'k3' wrote /workspace/data/experiments/pairs_k_comparison/manifests/k3/val.jsonl (32 rows)
2026-04-01 09:19:51.627 | INFO     | aitraf.data_ops.create_pairwise_manifests:_build_task_manifests:161 - Task 'k3' wrote /workspace/data/experiments/pairs_k_comparison/manifests/k3/test.jsonl (31 rows)


,split,rows
0,train,247
1,val,32
2,test,31


## Step 3. Simulate smaller `k` values on the observed `train` and `val` graphs

We build one deterministic opponent ordering per clip from the observed `k=3` split graph. Then:
- `k=3`: keep the full observed split as-is
- `k=2`: keep pairs where either endpoint ranks the other in its top 2
- `k=1`: keep pairs where either endpoint ranks the other in its top 1

This makes the simulated subsets nested: `k1 ⊂ k2 ⊂ k3`.


In [3]:
def canonical_pair(left: str, right: str) -> tuple[str, str]:
    return (left, right) if left <= right else (right, left)


def stable_rank_key(clip: str, opponent: str, seed: int) -> str:
    payload = f"{seed}|{clip}|{opponent}".encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def build_clip_opponent_orders(
    split_df: pd.DataFrame, seed: int
) -> dict[str, list[str]]:
    opponents: dict[str, set[str]] = {}
    for row in split_df.itertuples(index=False):
        left = row.left_s3_path
        right = row.right_s3_path
        opponents.setdefault(left, set()).add(right)
        opponents.setdefault(right, set()).add(left)

    ordered: dict[str, list[str]] = {}
    for clip, values in opponents.items():
        ordered[clip] = sorted(
            values, key=lambda other: stable_rank_key(clip, other, seed)
        )
    return ordered


def build_split_subset(split_df: pd.DataFrame, k: int, seed: int) -> pd.DataFrame:
    if k == 3:
        return split_df.reset_index(drop=True)

    opponent_orders = build_clip_opponent_orders(split_df, seed)
    allowed_pairs: set[tuple[str, str]] = set()

    for clip, opponents in opponent_orders.items():
        for opponent in opponents[:k]:
            allowed_pairs.add(canonical_pair(clip, opponent))

    subset_df = split_df.loc[
        split_df.apply(
            lambda row: canonical_pair(row["left_s3_path"], row["right_s3_path"])
            in allowed_pairs,
            axis=1,
        )
    ].reset_index(drop=True)
    return subset_df


simulated_splits = {
    k: {
        "train": build_split_subset(k3_splits["train"], k, EDGE_ORDER_SEED),
        "val": build_split_subset(k3_splits["val"], k, EDGE_ORDER_SEED),
        "test": k3_splits["test"].reset_index(drop=True),
    }
    for k in K_VALUES
}

display(
    pd.DataFrame(
        [
            {"k": k, "split": split, "rows": len(df)}
            for k, split_map in simulated_splits.items()
            for split, df in split_map.items()
        ]
    )
    .sort_values(["k", "split"])
    .reset_index(drop=True)
)

,k,split,rows
0,1,test,31
1,1,train,73
2,1,val,30
3,2,test,31
4,2,train,134
5,2,val,32
6,3,test,31
7,3,train,247
8,3,val,32


## Step 4. Save the simulated manifests

We write `train/val/test` for `k=1` and `k=2`, and keep the already-created `k=3` manifests.


In [4]:
for k in [1, 2]:
    manifest_dir = MANIFESTS_DIR / f"k{k}"
    manifest_dir.mkdir(parents=True, exist_ok=True)

    for split in ["train", "val", "test"]:
        simulated_splits[k][split].to_json(
            manifest_dir / f"{split}.jsonl",
            orient="records",
            lines=True,
            force_ascii=False,
        )

    shutil.copy2(MANIFESTS_DIR / "k3" / "vocab.json", manifest_dir / "vocab.json")

## Step 5. Quick sanity check

This shows one overall comparison table for `k=1`, `k=2`, and `k=3`.


In [5]:
rows = []

for k in K_VALUES:
    manifest_dir = MANIFESTS_DIR / f"k{k}"
    split_dfs = {
        split: pd.DataFrame(
            ScorePredictionRankDataset(
                manifests_dir=manifest_dir, split=split
            ).manifest_rows()
        )
        for split in ["train", "val", "test"]
    }
    all_df = pd.concat(split_dfs.values(), ignore_index=True)
    rows.append(
        {
            "k": k,
            "all_rows": len(all_df),
            "train_rows": len(split_dfs["train"]),
            "val_rows": len(split_dfs["val"]),
            "test_rows": len(split_dfs["test"]),
        }
    )

display(pd.DataFrame(rows).sort_values(["k"]).reset_index(drop=True))

,k,all_rows,train_rows,val_rows,test_rows
0,1,134,73,30,31
1,2,197,134,32,31
2,3,310,247,32,31


## Step 6. Rank pair-count stats for each `k`

This shows how many times clips appear in the full ranked rows for each `k`, summarized by min, mean, and max.


In [6]:
rank_count_rows = []

for k in K_VALUES:
    manifest_dir = MANIFESTS_DIR / f"k{k}"
    split_frames = []
    for split in ["train", "val", "test"]:
        manifest_df = pd.DataFrame(
            ScorePredictionRankDataset(
                manifests_dir=manifest_dir, split=split
            ).manifest_rows()
        )
        split_rank_df = ao_soul_pairwise_labels_df.loc[
            ao_soul_pairwise_labels_df["annotation_id"].isin(
                manifest_df["annotation_id"]
            )
        ]
        split_frames.append(split_rank_df)

    rank_df = pd.concat(split_frames, ignore_index=True)
    clip_counts = (
        pd.concat([rank_df["left"], rank_df["right"]])
        .value_counts()
        .rename_axis("clip")
        .rename("pair_count")
    )
    rank_count_rows.append(
        {
            "k": k,
            "ranks": len(rank_df),
            "clips": int(clip_counts.shape[0]),
            "min_pair_count": int(clip_counts.min()),
            "mean_pair_count": float(clip_counts.mean()),
            "max_pair_count": int(clip_counts.max()),
        }
    )

display(pd.DataFrame(rank_count_rows).sort_values(["k"]).reset_index(drop=True))

,k,clips,min_pair_count,mean_pair_count,max_pair_count
0,1,80,1,1.825,5
1,2,80,2,3.350,8
2,3,80,3,6.175,11
